# Module `metric_closure.py`

Ce notebook explique la fermeture metrique d'un graphe. Cette brique est volontairement separee du generateur, car elle sera reutilisee par :

- les solveurs qui ont besoin d'une matrice complete entre sommets ;
- la simulation dynamique qui doit recalculer la matrice a chaque tour ;
- l'execution dynamique `execute_dynamic` qui trace le chemin reel pris par un vehicule.

Le module contient des primitives reutilisables et une fonction haut-niveau.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.metric_closure import (
    build_cost_matrix,
    build_neighbor_map,
    check_triangle_inequality,
    complete_graph_with_shortest_paths,
    dijkstra,
    normalize_edge,
    reconstruct_path,
)

## 1. Role du module

Le graphe residuel peut avoir des paires de sommets sans arete directe. Pour un solveur TSP/VRP metrique, on veut pourtant un cout entre chaque paire.

`metric_closure.py` produit :

- `completed_costs` : matrice `n x n` des plus courts chemins ;
- `completed_paths` : dictionnaire `{(i, j): [i, ..., j]}` qui donne la vraie sequence de sommets.

La matrice respecte l'inegalite triangulaire (consequence directe de Dijkstra), ce qui permet d'appliquer des heuristiques metriques sans garanties cassees.

## 2. `normalize_edge` : cle canonique

Une arete non orientee est identifiee par `(min(u, v), max(u, v))`. `normalize_edge` formalise cette convention.

Consequence : partout dans le code, lire ou ecrire une arete se fait avec cette cle. Cela evite les doublons `(0, 3)` et `(3, 0)`.

In [ ]:
print(normalize_edge(3, 0))
print(normalize_edge(0, 3))
print(normalize_edge(4, 4))

## 3. `build_cost_matrix` : dictionnaire -> matrice dense

Prend un `dict[(u,v), cout]` et produit la matrice symetrique `n x n`. Les paires absentes ou a cout `inf` laissent la case a `0.0`.

Utilite : donner au reste du code une structure indexable par `matrix[i][j]` sans chercher dans un dict.

In [ ]:
edge_costs = {(0, 1): 4.0, (1, 2): 3.0, (0, 2): 10.0}
for row in build_cost_matrix(3, edge_costs):
    print(row)

## 4. `build_neighbor_map` : format adapte a Dijkstra

Transforme le dict d'aretes en listes d'adjacence : `{sommet: [(voisin, cout), ...]}`.

Les aretes a cout `inf` (FORBIDDEN) sont filtrees a la construction. Resultat : Dijkstra ne les voit jamais.

In [ ]:
neighbors = build_neighbor_map(3, edge_costs)
for node, nbrs in neighbors.items():
    print(node, "->", nbrs)

## 5. `dijkstra` : plus courts chemins depuis une source

Implementation classique a tas binaire (`heapq`) :

- file de priorite sur `(cout_cumule, sommet)` ;
- on ignore un pop si `current_cost > distances[node]` (entree perimee) ;
- on met a jour `predecessors[neighbor]` pour reconstruire les chemins plus tard.

Complexite : O((n + m) log n). Suffisant pour les tailles cibles (jusqu'a quelques centaines de sommets).

In [ ]:
distances, predecessors = dijkstra(0, 3, neighbors)
print("Distances depuis 0 :", distances)
print("Predecesseurs     :", predecessors)

## 6. `reconstruct_path` : remonter la trace

A partir du tableau `predecessors`, on reconstruit le chemin `source -> target` en remontant de `target` vers `source`.

Cas particuliers :

- `source == target` : renvoie `[source]`.
- `target` inatteignable : renvoie `[]` (distance infinie, pas de predecesseur).

Ce tableau de predecesseurs est produit a chaque Dijkstra ; `reconstruct_path` est juste un accesseur.

In [ ]:
print("Chemin 0 -> 2 :", reconstruct_path(0, 2, predecessors))
print("Chemin 0 -> 0 :", reconstruct_path(0, 0, predecessors))

## 7. `complete_graph_with_shortest_paths` : haut niveau

Boucle Dijkstra depuis chaque sommet et construit les deux livrables :

- `matrix[source][target] = plus court cout` ;
- `paths[(s, t)] = [s, ..., t]` pour `s < t` (paires normalisees).

Cette fonction est le **seul** appel que le generateur et le simulateur dynamique font en pratique.

In [ ]:
node_count = 4
edge_costs = {(0, 1): 4.0, (1, 2): 3.0, (2, 3): 5.0, (0, 2): 10.0}

completed_costs, completed_paths = complete_graph_with_shortest_paths(node_count, edge_costs)
print("Matrice complete :")
for row in completed_costs:
    print(row)

print("\nChemins reels :")
for key, path in sorted(completed_paths.items()):
    print(f"  {key} -> {path}")

## 8. `check_triangle_inequality` : garde-fou

Verifie que `d(i, j) <= d(i, k) + d(k, j)` pour tous les triplets. Si la propriete casse (rare avec Dijkstra, mais peut arriver avec une matrice chargee manuellement), la fonction renvoie `(False, (i, j, k))` et l'appelant sait exactement ou regarder.

La tolerance `1e-9` absorbe les erreurs d'arrondi.

In [ ]:
ok, violation = check_triangle_inequality(completed_costs)
print("Inegalite respectee ?", ok)
print("Violation         :", violation)

## 9. Pourquoi un module dedie ?

Isoler Dijkstra et la fermeture metrique rend l'architecture plus souple. Une future version peut remplacer l'implementation par :

- A* si on ajoute des heuristiques geographiques ;
- un backend C si les performances deviennent critiques ;
- un cache incremental si la dynamique ne modifie que peu d'aretes a la fois.

Toutes ces evolutions se feraient sans toucher au generateur ni aux solveurs.